# UIT DSC 2026 LegalQA dưới 4B

Pipeline quality V8: BM25 thường + BM25 cụm từ/chính xác + E5 small → RRF → Vietnamese Reranker kèm legal/recency boosts → ngữ cảnh điều luật → Vi-Qwen2-3B-RAG, bắt buộc QLoRA và chọn checkpoint theo METEOR.

Tổng dự kiến **3.801.281.793 tham số** khi gắn LoRA rank 16. Cell khóa mô hình đếm lại bằng kiến trúc đầy đủ.

Notebook dùng cùng cấu hình quality V8 với smoke30 và dev100, clone mã nguồn từ GitHub vào Kaggle và tái sử dụng full index/model từ Dataset Version 3; không chạy theo đường dẫn máy local. QLoRA dùng tập con train xác định bởi seed và lexical retrieval nhanh, sau đó đánh giá từng checkpoint trên dev100.

Chạy tuần tự từng cell. Gắn hai Dataset `lighth/ver3-smoke-output` và `lighth/uit-dsc-2026-task2-legalqa-train`, đồng thời bật GPU và Internet. Mọi inference diễn ra trong Kaggle runtime. Các cell SFT/holdout/submission chỉ chạy khi bật cờ tương ứng ở cell 1.

## 1 Thiết lập Kaggle và đường dẫn

Notebook chỉ chạy trên Kaggle. Cell 2 clone nhánh `main` của repo GitHub vào `/kaggle/working/uit-dsc-2026-task2-legalqa`, đọc dữ liệu từ Dataset train và dùng lại index/model trong Dataset `ver3-smoke-output`.

Giữ `USE_REPO_DATA = False` cho public/dev. Nếu chạy private, trỏ `KAGGLE_DATASET_ROOT` tới Dataset chứa đúng `train.json` và test tương ứng, rồi đổi `PHASE` thành `private`.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

KAGGLE_ROOT = Path('/kaggle')
if not KAGGLE_ROOT.exists():
    raise RuntimeError('Notebook này được cấu hình chỉ để chạy trên Kaggle.')

WORK_BASE = Path('/kaggle/working')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_REF = 'main'
CODE = WORK_BASE / 'uit-dsc-2026-task2-legalqa'
VERSION3_URL = 'https://www.kaggle.com/datasets/lighth/ver3-smoke-output'
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
SAVED_INDEX = VERSION3_ROOT / 'index'
SAVED_MODELS = VERSION3_ROOT / 'models'

USE_REPO_DATA = False
KAGGLE_DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')
DATASET_ROOT = CODE if USE_REPO_DATA else KAGGLE_DATASET_ROOT
TRAIN_PATH = DATASET_ROOT / 'train.json'
TEST_PATH = DATASET_ROOT / 'public-official.json'
PHASE = 'public'

RUN_ROOT = WORK_BASE / 'legalqa_quality_v8_full'
MODELS = RUN_ROOT / 'models'
INDEX = SAVED_INDEX
DATA = RUN_ROOT / f'data_{PHASE}'
CFG = RUN_ROOT / 'config.json'

RUN_SFT = True          # Bắt buộc trong V8; notebook sẽ dừng nếu tắt.
RUN_HOLDOUT = False     # Bật sau khi chốt checkpoint/cấu hình bằng dev.
RUN_SUBMISSION = False  # Bật khi muốn sinh toàn bộ test và đóng ZIP.
RESUME_CHECKPOINT = None  # Ví dụ: RUN_ROOT / 'sft' / 'checkpoint-350'

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_ROOT, '| Phase:', PHASE)

## 2 Clone mã nguồn từ GitHub

Cell này clone repo mới vào `/kaggle/working`. Khi chạy lại, notebook chỉ cập nhật bằng fast-forward để không ghi đè thay đổi cục bộ trong phiên Kaggle. Muốn đổi hyperparameter, sửa `RUN_ROOT/config.json` trước các stage; dùng output mới nếu thay đổi làm cache fingerprint khác.

In [ ]:
if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} đã tồn tại nhưng không phải Git repo. Hãy Restart Session hoặc đổi thư mục CODE.')
    current_remote = subprocess.check_output(
        ['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if current_remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError(f'Remote hiện tại không đúng repo yêu cầu: {current_remote}')
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CODE)],
        check=True,
    )

for label, path in [('train', TRAIN_PATH), ('test', TEST_PATH)]:
    if not path.exists():
        raise FileNotFoundError(f'{label}: {path}. Sửa cấu hình dữ liệu tại cell 1.')

required_index = ['index_manifest.json', 'corpus.sqlite', 'dense.faiss']
missing_index = [name for name in required_index if not (SAVED_INDEX / name).is_file()]
missing_models = [name for name in ['models.lock.json'] if not (SAVED_MODELS / name).is_file()]
missing_roles = [role for role in ['embedding', 'reranker', 'generator'] if not (SAVED_MODELS / role / 'config.json').is_file()]
if missing_index or missing_models or missing_roles:
    raise FileNotFoundError(
        'Thiếu artifacts Version 3. Hãy Add Input dataset lighth/ver3-smoke-output từ ' + VERSION3_URL
        + f' | index={missing_index}, model_files={missing_models}, model_roles={missing_roles}'
    )

MODELS.mkdir(parents=True, exist_ok=True)
for role in ['embedding', 'reranker', 'generator']:
    source_dir = SAVED_MODELS / role
    link = MODELS / role
    if link.exists() or link.is_symlink():
        if link.resolve() != source_dir.resolve():
            raise RuntimeError(f'Symlink model không đúng: {link} -> {link.resolve()}')
    else:
        link.symlink_to(source_dir, target_is_directory=True)
shutil.copy2(SAVED_MODELS / 'models.lock.json', MODELS / 'models.lock.json')

shared_cfg = json.loads((CODE / 'config.json').read_text(encoding='utf-8'))
if not shared_cfg['training'].get('required') or not RUN_SFT:
    raise RuntimeError('Quality V8 bắt buộc chạy QLoRA; không được tắt RUN_SFT hoặc training.required.')
if not shared_cfg['generation'].get('load_in_4bit'):
    raise RuntimeError('QLoRA bắt buộc load generator ở NF4 4-bit.')
if shared_cfg['training'].get('retrieval_mode') != 'lexical':
    raise RuntimeError('Training retrieval phải dùng lexical mode đã kiểm định để kịp giới hạn Kaggle.')
if shared_cfg['evaluation'].get('primary_metric') != 'meteor' or shared_cfg['evaluation'].get('target_meteor') != 0.65:
    raise RuntimeError('Quality V8 phải ưu tiên METEOR với mục tiêu 0.65.')
CFG.write_text(json.dumps(shared_cfg, ensure_ascii=False, indent=2), encoding='utf-8')
print('Shared retrieval config:', shared_cfg['retrieval'])
print('Shared generation config:', shared_cfg['generation'])
print('Evaluation objective:', shared_cfg['evaluation'])

def run(*args):
    command = [sys.executable, '-m', 'legalqa', '--config', str(CFG), '--models', str(MODELS), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=CODE, check=True)

commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Code:', CODE, '| Commit:', commit)
print('Hướng dẫn:', CODE / 'README.md')

## 3 Cài môi trường và kiểm định CPU

Dependencies chỉ được import trong subprocess để tránh module cũ của kernel. WordNet được dùng đúng vai trò metric BTC, không thêm vào dữ liệu train. Self-test metric dùng chuỗi thử nhỏ không đi vào pipeline.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, check=True)
subprocess.run([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, check=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_ROOT / 'environment.freeze.txt').write_text(freeze, encoding='utf-8')

## 4 Chia tập và audit mô hình tái sử dụng

QA trùng question hoặc answer chuẩn hóa được giữ cùng split. Dev/holdout được chia theo nhóm nên kích thước xấp xỉ 10% mỗi tập. Model weights và lock được đọc từ Dataset Version 3; audit luôn tính tham số gốc và LoRA, không tính theo số byte 4-bit.

In [ ]:
run('prepare', '--train', TRAIN_PATH, '--test', TEST_PATH, '--output', DATA)
run('audit-models')
audit = json.loads((MODELS / 'parameter_audit.json').read_text())
assert audit['passes'] and audit['total_with_unmerged_lora'] < 4_000_000_000
print('Total:', f"{audit['total_with_unmerged_lora']:,}")
print(json.loads((DATA / 'data_report.json').read_text()))

## 5 Xác nhận full index Version 3

Notebook chỉ đọc full index 407.107 chunks từ Dataset Version 3. Không chạy lại corpus preparation hoặc embeddings. Các thay đổi quality V8 tác động query retrieval/reranking, đóng gói prompt, fallback cục bộ và QLoRA; chúng tương thích với index hiện có.

In [ ]:
manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
if manifest.get('chunks') != 407_107:
    raise RuntimeError(f"Không phải full index Version 3: chunks={manifest.get('chunks')}")
if manifest.get('documents') != 8_507:
    raise RuntimeError(f"Số documents không khớp Version 3: documents={manifest.get('documents')}")
print('Reusing completed Version 3 index:', INDEX)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 6 Smoke 30 câu

Đây là lần kiểm tra đầu tiên cần trọng số thật. Đọc một vài answer và audit trước khi chạy dài. Điểm 30 câu không đủ để kết luận model nào tốt hơn.

In [ ]:
SMOKE_CACHE = RUN_ROOT / 'dev30.retrieval.json'
SMOKE_PRED = RUN_ROOT / 'dev30.base.json'
run('retrieve', '--questions', DATA / 'dev30.questions.json', '--index', INDEX, '--output', SMOKE_CACHE)
run('generate', '--questions', DATA / 'dev30.questions.json', '--retrieval', SMOKE_CACHE, '--output', SMOKE_PRED)
run('evaluate', '--predictions', SMOKE_PRED, '--references', DATA / 'dev30.references.json', '--output', RUN_ROOT / 'dev30.base.metrics.json', '--label', 'base_smoke')
qa = json.loads((DATA / 'dev30.json').read_text())
predictions = json.loads(SMOKE_PRED.read_text())
audits = json.loads(SMOKE_PRED.with_suffix('.audit.json').read_text())
for key in list(predictions)[:3]:
    print('\nID:', key, '|', qa[key]['question'])
    print('Prediction:', predictions[key]['answer'])
    print('Reference:', qa[key]['answer'])
    print('Audit:', {k: v for k, v in audits[key].items() if k != 'raw_answer'})

## 7 Baseline và truy xuất tập phát triển

Chạy baseline trên split `training.selection_split` (mặc định dev100) để so sánh trực tiếp V5–V8 và giảm thời gian chọn checkpoint. Tập QLoRA được chọn xác định bằng seed, giới hạn bởi `training.max_examples`, rồi tạo context bằng lexical retrieval không tải encoder/reranker. Đây chỉ là cache train; inference dev/test vẫn dùng retrieval đầy đủ.

In [ ]:
SELECTION_SPLIT = shared_cfg['training']['selection_split']
SELECTION_QUESTIONS = DATA / f'{SELECTION_SPLIT}.questions.json'
SELECTION_REFERENCES = DATA / f'{SELECTION_SPLIT}.references.json'
DEV_CACHE = RUN_ROOT / f'{SELECTION_SPLIT}.retrieval.json'
BASE_PRED = RUN_ROOT / f'{SELECTION_SPLIT}.base.json'
BASE_REPORT = RUN_ROOT / f'{SELECTION_SPLIT}.base.metrics.json'
SFT_TRAIN = DATA / 'train.sft.json'
SFT_QUESTIONS = DATA / 'train.sft.questions.json'
TRAIN_CACHE = RUN_ROOT / 'train.sft.lexical.retrieval.json'
run('retrieve', '--questions', SELECTION_QUESTIONS, '--index', INDEX, '--output', DEV_CACHE)
run('generate', '--questions', SELECTION_QUESTIONS, '--retrieval', DEV_CACHE, '--output', BASE_PRED)
run('evaluate', '--predictions', BASE_PRED, '--references', SELECTION_REFERENCES, '--output', BASE_REPORT, '--label', 'base_v8')
run('prepare-sft', '--train', DATA / 'train.json', '--output', SFT_TRAIN)
sft_questions = json.loads(SFT_QUESTIONS.read_text(encoding='utf-8'))
assert len(sft_questions) == min(shared_cfg['training']['max_examples'], json.loads((DATA / 'data_report.json').read_text())['split_sizes']['train'])
run('retrieve', '--questions', SFT_QUESTIONS, '--index', INDEX, '--output', TRAIN_CACHE, '--mode', shared_cfg['training']['retrieval_mode'])
print('QLoRA examples:', len(sft_questions), '| retrieval mode:', shared_cfg['training']['retrieval_mode'])

## 8 Fine tune

QLoRA là bước bắt buộc của V8, huấn luyện trên answer gốc và EOS; prompt bị mask khỏi loss. Tập con 768 mẫu được chọn tái lập bằng seed, prompt train tối đa 2.048 token để cân bằng chất lượng với thời lượng Kaggle; inference vẫn dùng 4.096 token. Mẫu không vừa toàn bộ target được ghi ID trong training_data_report.json. CLI chỉ cho Trainer thấy GPU 0. Khi hết phiên, Save Version rồi dùng checkpoint resume; không tự tắt QLoRA hoặc coi GPU 2×T4 là một GPU 32GB.

In [ ]:
SFT_DIR = RUN_ROOT / 'sft'
args = ['fit', '--train', SFT_TRAIN, '--retrieval', TRAIN_CACHE, '--output', SFT_DIR, '--gpu', '0']
if RESUME_CHECKPOINT is not None:
    args += ['--resume', RESUME_CHECKPOINT]
if (SFT_DIR / 'adapter_last' / 'adapter_config.json').exists() and RESUME_CHECKPOINT is None:
    print('Required QLoRA already completed; evaluating saved epoch checkpoints.')
else:
    run(*args)
if not (SFT_DIR / 'training_result.json').is_file():
    raise RuntimeError('QLoRA did not complete; do not continue to selection/submission.')
print('QLoRA result:', json.loads((SFT_DIR / 'training_result.json').read_text(encoding='utf-8')))

## 9 Chọn checkpoint trên dev

Đánh giá bắt buộc từng checkpoint epoch trên cùng dev100, kể cả khi chúng được tạo ở phiên trước. Baseline chỉ dùng để so sánh, không được thay thế adapter trong main run. Chọn checkpoint QLoRA theo METEOR; ROUGE-L chỉ phá hòa. Nếu không có checkpoint hoặc adapter hợp lệ, notebook dừng trước holdout/submission.

In [ ]:
reports = []
checkpoints = sorted(SFT_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1])) if SFT_DIR.exists() else []
if not checkpoints:
    raise RuntimeError('Không tìm thấy checkpoint QLoRA để đánh giá.')
for checkpoint in checkpoints:
    pred_path = RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.json'
    report_path = RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.metrics.json'
    run('generate', '--questions', SELECTION_QUESTIONS, '--retrieval', DEV_CACHE, '--adapter', checkpoint, '--output', pred_path)
    run('evaluate', '--predictions', pred_path, '--references', SELECTION_REFERENCES, '--output', report_path, '--label', checkpoint.name)
    run('compare', '--baseline', BASE_REPORT, '--candidate', report_path, '--output', RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.comparison.json')
    reports.append(report_path)
run('select', '--reports', *reports, '--output', RUN_ROOT / 'selection.json')
selection = json.loads((RUN_ROOT / 'selection.json').read_text())
SELECTED_ADAPTER = selection['prediction_manifest']['adapter_path']
if not SELECTED_ADAPTER:
    raise RuntimeError('Main run bắt buộc dùng checkpoint QLoRA, không được fallback về baseline.')
base_metrics = json.loads(BASE_REPORT.read_text(encoding='utf-8'))
if selection['meteor'] < base_metrics['meteor']:
    print('WARNING: QLoRA tốt nhất vẫn thấp hơn baseline; main vẫn giữ adapter theo yêu cầu bắt buộc.')
print('Selected:', selection['label'], '| adapter:', SELECTED_ADAPTER)
print('Primary objective:', selection['objective'])

## 10 Holdout

Bật sau khi đã khóa cấu hình. Giữ nguyên lựa chọn từ dev. Nếu dùng kết quả holdout để sửa tiếp, tập này trở thành development data, không còn là phép đo độc lập.

In [ ]:
if RUN_HOLDOUT:
    cache = RUN_ROOT / 'holdout.retrieval.json'
    pred = RUN_ROOT / 'holdout.selected.json'
    run('retrieve', '--questions', DATA / 'holdout.questions.json', '--index', INDEX, '--output', cache)
    args = ['generate', '--questions', DATA / 'holdout.questions.json', '--retrieval', cache, '--output', pred, '--adapter', SELECTED_ADAPTER]
    run(*args)
    run('evaluate', '--predictions', pred, '--references', DATA / 'holdout.references.json', '--output', RUN_ROOT / 'holdout.selected.metrics.json', '--label', selection['label'])
else:
    print('Holdout chưa chạy.')

## 11 Public hoặc private submission

Chỉ tạo predictions, không tự gửi lên Codabench. Mã chấm yêu cầu tập IDs trùng chính xác và các giá trị dạng `{answer: string}`. ZIP mặc định có submission.json ở gốc. Tài liệu và ZIP đính kèm thiếu metadata reference xác nhận tên file input phía server; nếu vòng thi quy định tên khác, đổi SUBMISSION_FILENAME dưới đây.

Save Version để giữ output, model lock, index, adapter và reports. Đây là bước lưu kết quả thực nghiệm cần thiết trước khi Kaggle hết phiên.

In [ ]:
SUBMISSION_FILENAME = 'submission.json'
if RUN_SUBMISSION:
    cache = RUN_ROOT / f'{PHASE}.retrieval.json'
    destination = RUN_ROOT / 'submissions' / PHASE
    destination.mkdir(parents=True, exist_ok=True)
    pred = destination / 'submission.json'
    run('retrieve', '--questions', DATA / 'test.questions.json', '--index', INDEX, '--output', cache)
    args = ['generate', '--questions', DATA / 'test.questions.json', '--retrieval', cache, '--output', pred, '--adapter', SELECTED_ADAPTER]
    run(*args)
    run('package', '--predictions', pred, '--questions', DATA / 'test.questions.json', '--output', destination / 'submission.zip', '--filename', SUBMISSION_FILENAME)
    print('Output:', destination / 'submission.zip')
    print('Reproducibility artifacts:', RUN_ROOT)
else:
    print('Submission chưa chạy. Bật RUN_SUBMISSION khi muốn sinh toàn bộ test.')